# CallSense — الملخص والحل (النسخة النهائية)
**المدخلات:** نص المكالمة + الفئة + قرار التصعيد (من المصنّف).
**المخرجات:** ملخص + حل احترافي + ملاحظة تسليم للمشرف عند التصعيد + تنبيهات للمراجعة.

**المعتمد:** البحث بـ `BAAI/bge-m3` + التوليد بـ `Qwen/Qwen2.5-7B-Instruct` (4-bit) + التصويت بين 3 مكالمات كتحقق إضافي على خارج النطاق.

**قبل البدء:** Runtime ← Change runtime type ← **T4 GPU**

**مكانه في النظام:** الموقع ← Whisper ← المصنّف ← **هذي المرحلة** ← الملخص والحل.

**الترتيب:** 1 ← Restart session ← 2 ← 2-ب ← 3 ← 4 ← 5 ← 6 ← 7 ← 8

## 1) تثبيت المكتبات
⚠️ بعد ما تخلص هذي الخلية: **Runtime ← Restart session** مرة وحدة، وبعدها كمّلوا من الخلية 2. ولا تعيدونها بعد إعادة التشغيل.

In [1]:
!pip install -q "pandas==2.2.3" sentence-transformers
!pip install -q -U bitsandbytes accelerate

## 2) تحميل البيانات
ارفعوا `Whisper_Small_Baseline_Results.xlsx`. نص المكالمة المستخدم هو مخرج Whisper لأنه اللي يتوفر للنظام فعليًا.

In [2]:
import pandas as pd, numpy as np
from google.colab import files

up = files.upload()
df = pd.read_excel(list(up.keys())[0], sheet_name="Results")

df["intent"] = df["intent"].str.strip().str.lower()                          # توحيد أسماء الفئات
df["reference_transcript"] = df["reference_transcript"].astype(str).str.replace("\\n", "\n")
kb = df.rename(columns={"whisper_prediction": "call_text"})
kb = kb[["call_id", "intent", "escalation", "call_text", "reference_transcript"]].reset_index(drop=True)
kb["call_text"] = kb["call_text"].astype(str)
kb["resolution"] = ""                                                        # الحلول المعتمدة تنضاف لاحقًا

print(len(kb), "مكالمة")
print(kb.intent.value_counts())

Saving Whisper_Small_Baseline_Results.xlsx to Whisper_Small_Baseline_Results.xlsx
80 مكالمة
intent
information     19
technical       14
cancellation    13
billing         12
complaint       12
out_of_scope    10
Name: count, dtype: int64


## 2-ب) التخزين الدائم في Google Drive
يربط Drive ويجهز ملفين في مجلد `CallSense`:
- `results_log.csv`: كل نتيجة يطلعها النظام، مع أعمدة المراجعة.
- `approved_kb.csv`: الحلول المعتمدة، وتنضاف لقاعدة البحث تلقائيًا كل مرة تشغّلون الملف.

In [3]:
import os
from google.colab import drive
drive.mount("/content/drive")

STORE_DIR = "/content/drive/MyDrive/CallSense"
LOG_PATH = os.path.join(STORE_DIR, "results_log.csv")
APPROVED_PATH = os.path.join(STORE_DIR, "approved_kb.csv")
os.makedirs(STORE_DIR, exist_ok=True)

KB_COLS = ["call_id", "intent", "escalation", "call_text", "reference_transcript", "resolution"]
if os.path.exists(APPROVED_PATH):
    approved = pd.read_csv(APPROVED_PATH, dtype=str).fillna("")
    kb = pd.concat([kb, approved[KB_COLS]], ignore_index=True)
    print("انضاف للقاعدة", len(approved), "حل معتمد")
kb = kb.fillna("").reset_index(drop=True)
print("حجم القاعدة:", len(kb), "| السجل:", LOG_PATH)

Mounted at /content/drive
حجم القاعدة: 80 | السجل: /content/drive/MyDrive/CallSense/results_log.csv


## 3) تحميل الموديلات
تاخذ 5–10 دقائق أول مرة.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline, BitsAndBytesConfig

EMB = "BAAI/bge-m3"
LLM = "Qwen/Qwen2.5-7B-Instruct"

emb_model = SentenceTransformer(EMB)
gen = pipeline("text-generation", model=LLM, device_map="auto",
               model_kwargs={"quantization_config": BitsAndBytesConfig(load_in_4bit=True)})

kb_emb = emb_model.encode(kb["call_text"].tolist(), convert_to_tensor=True, normalize_embeddings=True)
print("جاهز:", EMB, "+", LLM)

## 4) البحث واكتشاف خارج النطاق
البحث يعطي أولوية للمكالمات من **نفس الفئة** اللي جات من المصنّف. والتصويت بين 3 مكالمات يشتغل كتحقق إضافي على قرار "خارج النطاق".

In [5]:
INTENT_AR = {"billing": "الفواتير", "technical": "مشكلة تقنية", "cancellation": "إلغاء",
             "complaint": "شكوى", "information": "استفسار", "out_of_scope": "خارج النطاق"}

OOS_REPLY = ("عذرًا، أنا مساعد متخصص في خدمات الاتصالات فقط، مثل الفواتير والباقات والمشاكل التقنية والإلغاء. "
             "ما أقدر أساعدك في هذا الطلب، لكن إذا عندك أي استفسار يخص خدمات الاتصالات أنا حاضر.")

def norm_intent(v):
    return str(v).strip().lower().replace(" ", "_")

def to_bool(v):
    return str(v).strip().lower() in ("1", "1.0", "true", "yes", "y", "نعم")

def retrieve(text, intent=None, exclude_id=None, k=3):
    q = emb_model.encode(text, convert_to_tensor=True, normalize_embeddings=True)
    sims = util.cos_sim(q, kb_emb)[0]
    rank = sims.clone()
    if intent is not None:
        same = torch.tensor((kb.intent == intent).values, device=rank.device)
        rank = rank + same.float()                       # نفس الفئة أولًا، والباقي يكمّل لو ما كفى
    for i in kb.index[kb.call_id == exclude_id]:
        rank[i] = float("-inf")                          # لا يسترجع نفس المكالمة
    idx = rank.topk(k).indices.tolist()
    return idx, [round(float(sims[i]), 3) for i in idx]

def vote_out_of_scope(text, exclude_id=None, k=3):
    idx, _ = retrieve(text, intent=None, exclude_id=exclude_id, k=k)
    return sum(kb.intent[i] == "out_of_scope" for i in idx) > k / 2

## 5) البرومبت ودالة الملخص والحل
**المدخلات:** نص المكالمة + الفئة + قرار التصعيد (من المصنّف).
**المخرجات:** ملخص + حل (موثّق إذا كان واضحًا، أو مقترح إذا كان ناقصًا) + ملاحظة تسليم للمشرف إذا كان التصعيد مطلوبًا + تنبيهات للمراجعة.

الموديل ما يغيّر الفئة ولا قرار التصعيد، لكن ينبّه إذا شاف الفئة ما تطابق المكالمة.

In [6]:
import json, re, time

SYS_DOC = """أنت خبير جودة في مركز خدمة عملاء شركة اتصالات سعودية. مهمتك كتابة ملخص المكالمة وحلها بشكل احترافي ليكون مرجعًا للموظفين والمشرفين.
القواعد:
- الفئة وقرار التصعيد يأتيان من نظام التصنيف. اعتمدهما ولا تغيّرهما.
- إذا رأيت أن الفئة لا تطابق محتوى المكالمة بوضوح، اذكر ذلك في intent_check مع السبب.
- اعتمد على نص المكالمة أولًا، ثم على المكالمات المشابهة المعطاة.
- إذا طبّق الموظف حلًا واضحًا، وثّقه كما حدث بدون إضافة معلومات غير موجودة.
- إذا كان الحل غير واضح أو ناقصًا، اقترح حلًا احترافيًا عمليًا مستفيدًا من المكالمات المشابهة.
- إذا كان التصعيد مطلوبًا، اجعل آخر خطوة في الحل هي التسليم للمشرف، واكتب escalation_handoff: المشكلة، وما تمت تجربته، والمطلوب من المشرف.
- إذا لم يكن التصعيد مطلوبًا، اترك escalation_handoff فارغًا.
- لا تخترع أرقامًا أو أسعارًا أو سياسات غير مذكورة في النصوص.
- اكتب بالعربية الفصحى المختصرة.
- أعد JSON فقط بدون أي نص قبله أو بعده."""

SCHEMA = """{
 "summary": "ملخص المشكلة في سطرين",
 "solution_status": "واضح" أو "جزئي" أو "غير موجود",
 "applied_solution": "ما فعله الموظف فعلًا، أو نص فارغ",
 "recommended_solution": ["خطوة 1", "خطوة 2"],
 "escalation_handoff": "ملاحظة التسليم للمشرف إذا كان التصعيد مطلوبًا، وإلا نص فارغ",
 "intent_check": "متوافق" أو "غير متوافق: السبب",
 "sources": ["أرقام المكالمات المشابهة التي استفدت منها فقط"],
 "confidence": "عالية" أو "متوسطة" أو "منخفضة"
}"""

def parse_json(txt):
    m = re.search(r"\{.*\}", txt, re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

def generate_digest(call_text, intent, escalation, call_id=None, k=3):
    """المدخلات من المصنّف: intent و escalation. المخرجات: dict فيه الملخص والحل والتنبيهات."""
    intent, esc = norm_intent(intent), to_bool(escalation)
    flags = []
    vote_oos = vote_out_of_scope(call_text, exclude_id=call_id, k=k)

    if intent == "out_of_scope":
        if not vote_oos:
            flags.append("المصنّف قال خارج النطاق لكن المكالمات المشابهة من خدمات الاتصالات")
        return {"call_id": call_id, "intent": intent, "escalation": esc,
                "summary": "طلب خارج نطاق خدمات الاتصالات", "solution_status": "خارج النطاق",
                "applied_solution": "", "recommended_solution": [OOS_REPLY], "escalation_handoff": "",
                "intent_check": "متوافق" if vote_oos else "يحتاج مراجعة", "sources": [], "retrieved": [],
                "confidence": "عالية" if vote_oos else "منخفضة", "flags": flags, "parse_ok": True, "raw": ""}

    if vote_oos:
        flags.append("المكالمات المشابهة تشير إلى أن الطلب قد يكون خارج النطاق")

    idx, sims = retrieve(call_text, intent=intent, exclude_id=call_id, k=k)
    retrieved = [kb.call_id[i] for i in idx]
    def case_text(i):
        t = f"[{kb.call_id[i]}] ({INTENT_AR.get(kb.intent[i], kb.intent[i])})\n{kb.reference_transcript[i]}"
        if str(kb.resolution[i]).strip():
            t += f"\nالحل المعتمد لهذه المكالمة:\n{kb.resolution[i]}"
        return t
    ctx = "\n\n".join(case_text(i) for i in idx)
    user = (f"معلومات من نظام التصنيف (اعتمدها كما هي):\n"
            f"- الفئة: {INTENT_AR.get(intent, intent)}\n"
            f"- التصعيد: {'مطلوب' if esc else 'غير مطلوب'}\n\n"
            f"نص المكالمة [{call_id or 'NEW'}]:\n{call_text}\n\n"
            f"مكالمات مشابهة سابقة:\n{ctx}\n\nأعد JSON بهذا الشكل:\n{SCHEMA}")
    out = gen([{"role": "system", "content": SYS_DOC}, {"role": "user", "content": user}],
              max_new_tokens=700, do_sample=False, repetition_penalty=1.1)
    raw = out[0]["generated_text"][-1]["content"]
    data = parse_json(raw)
    d = data or {}

    sources = d.get("sources") if isinstance(d.get("sources"), list) else []
    if data is None:
        flags.append("الموديل ما رجع JSON صحيح")
    if not all(s in retrieved for s in sources):
        flags.append("استشهد بمكالمات غير مسترجعة")
    if str(d.get("intent_check", "")).startswith("غير"):
        flags.append("الموديل يشك في الفئة: " + str(d.get("intent_check")))
    if esc and not str(d.get("escalation_handoff", "")).strip():
        flags.append("التصعيد مطلوب لكن ما فيه ملاحظة تسليم")

    return {"call_id": call_id, "intent": intent, "escalation": esc,
            "summary": d.get("summary", ""), "solution_status": d.get("solution_status", ""),
            "applied_solution": d.get("applied_solution", ""),
            "recommended_solution": d.get("recommended_solution", []),
            "escalation_handoff": d.get("escalation_handoff", ""), "intent_check": d.get("intent_check", ""),
            "sources": sources, "retrieved": retrieved, "confidence": d.get("confidence", ""),
            "flags": flags, "parse_ok": data is not None, "raw": "" if data else raw}

print("جاهز")

جاهز


## 6) تجربة سريعة قبل التشغيل الكامل
هذا شكل الاستدعاء اللي بيستخدمه نظامكم: نص المكالمة + الفئة + التصعيد من المصنّف.

In [ ]:
def show(res):
    print(json.dumps(res, ensure_ascii=False, indent=2))

row = kb[kb.call_id == "CS003"].iloc[0]
show(generate_digest(row.call_text, intent="billing", escalation=0, call_id=row.call_id))

print("\n" + "-" * 40 + "\n")
show(generate_digest("أبي أحجز موعد عند دكتور أسنان بكرة", intent="out_of_scope", escalation=0))

## 7) شكل المخرج النهائي للموقع
`generate_digest` ترجع dict (يتحول لـ JSON للموقع)، و `format_report` تحوله لنص عربي مرتب يُعرض للموظف أو المشرف، ويبدأ بالتصنيف اللي وصل من المصنّف.

In [ ]:
ESC_AR = {True: "مطلوب", False: "غير مطلوب"}

def format_report(res):
    lines = ["📋 تقرير المكالمة", "",
             f"التصنيف (من المصنّف): {INTENT_AR.get(res['intent'], res['intent'])}",
             f"التصعيد (من المصنّف): {ESC_AR[bool(res['escalation'])]}", "",
             "الملخص:", str(res["summary"]), ""]
    if res["solution_status"] == "خارج النطاق":
        lines += ["الرد:", OOS_REPLY]
    else:
        lines.append(f"الحل ({res['solution_status']}):")
        steps = res["recommended_solution"] if isinstance(res["recommended_solution"], list) else [res["recommended_solution"]]
        lines += [f"{n}. {s}" for n, s in enumerate(steps, 1)]
        if res["applied_solution"]:
            lines += ["", "ما طبّقه الموظف:", str(res["applied_solution"])]
        if res["escalation_handoff"]:
            lines += ["", "ملاحظة التسليم للمشرف:", str(res["escalation_handoff"])]
        if res["sources"]:
            lines += ["", "مبني على مكالمات: " + "، ".join(map(str, res["sources"]))]
        lines.append(f"درجة الثقة: {res['confidence']}")
    if res["flags"]:
        lines += ["", "⚠️ يحتاج مراجعة:"] + [f"- {f}" for f in res["flags"]]
    return "\n".join(lines)

# مثال: هذا اللي يصير لما يوصل نص Whisper + مخرج المصنّف من الموقع
call_text = kb[kb.call_id == "CS003"].iloc[0].call_text     # في الموقع: مخرج Whisper للتسجيل المرفوع
res = generate_digest(call_text, intent="billing", escalation=0)
print(format_report(res))

## 8) الحفظ والمراجعة
- `save_result`: تحفظ النتيجة في السجل وترجع رقمها. الموقع يستدعيها بعد كل مكالمة.
- `pending`: تعرض النتائج اللي لسا ما انراجعت.
- `approve`: تعتمد الحل (أو نسخة معدّلة منه)، وتضيفه لقاعدة البحث فورًا وفي Drive.
- `reject`: ترفض الحل مع سبب، وما ينضاف للقاعدة.

In [ ]:
import uuid
from datetime import datetime

LOG_COLS = ["record_id", "timestamp", "call_text", "intent", "escalation", "summary", "solution_status",
            "applied_solution", "recommended_solution", "escalation_handoff", "intent_check", "sources",
            "retrieved", "confidence", "flags", "review_status", "final_resolution", "reviewer_notes"]

def j(v):
    return "\n".join(map(str, v)) if isinstance(v, list) else str(v or "")

def load_log():
    return pd.read_csv(LOG_PATH, dtype=str).fillna("") if os.path.exists(LOG_PATH) else pd.DataFrame(columns=LOG_COLS)

def save_result(res, call_text):
    rid = "R" + datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:4]
    row = {"record_id": rid, "timestamp": datetime.now().isoformat(timespec="seconds"), "call_text": call_text,
           "intent": res["intent"], "escalation": int(res["escalation"]),
           **{k: j(res[k]) for k in ["summary", "solution_status", "applied_solution", "recommended_solution",
                                     "escalation_handoff", "intent_check", "sources", "retrieved", "confidence", "flags"]},
           "review_status": "", "final_resolution": "", "reviewer_notes": ""}
    pd.DataFrame([row], columns=LOG_COLS).to_csv(LOG_PATH, mode="a", index=False,
                                                 header=not os.path.exists(LOG_PATH), encoding="utf-8-sig")
    return rid

def pending():
    log = load_log()
    return log[log.review_status == ""][["record_id", "timestamp", "intent", "summary", "solution_status", "confidence", "flags"]]

def _set_review(rid, status, final="", notes=""):
    log = load_log()
    m = log.record_id == rid
    if not m.any():
        raise ValueError(f"ما لقيت النتيجة {rid}")
    log.loc[m, ["review_status", "final_resolution", "reviewer_notes"]] = [status, final, notes]
    log.to_csv(LOG_PATH, index=False, encoding="utf-8-sig")
    return log[m].iloc[0]

def approve(rid, edited_resolution="", notes=""):
    global kb, kb_emb
    log = load_log()
    m = log.record_id == rid
    if not m.any():
        raise ValueError(f"ما لقيت النتيجة {rid}")
    row = log[m].iloc[0]
    if row.review_status in ("معتمد", "معدّل"):
        print(f"{rid} معتمد من قبل")
        return
    edited = edited_resolution.strip()
    final = edited or row.recommended_solution
    _set_review(rid, "معدّل" if edited else "معتمد", final, notes)
    entry = pd.DataFrame([{"call_id": rid, "intent": row.intent, "escalation": row.escalation, "call_text": row.call_text,
                           "reference_transcript": row.call_text, "resolution": final}])
    entry.to_csv(APPROVED_PATH, mode="a", index=False, header=not os.path.exists(APPROVED_PATH), encoding="utf-8-sig")
    kb = pd.concat([kb, entry], ignore_index=True).fillna("")
    new_emb = emb_model.encode(row.call_text, convert_to_tensor=True, normalize_embeddings=True)
    kb_emb = torch.cat([kb_emb, new_emb.unsqueeze(0)])
    print(f"✓ {rid} انعتمد وانضاف للقاعدة | حجمها الحين: {len(kb)}")

def reject(rid, notes=""):
    _set_review(rid, "مرفوض", "", notes)
    print(f"✗ {rid} انرفض")

### مثال: الدورة كاملة
هذا اللي بيصير في الموقع: نتيجة ← حفظ ← مراجعة ← اعتماد.

In [ ]:
call_text = kb[kb.call_id == "CS003"].iloc[0].call_text      # في الموقع: مخرج Whisper
res = generate_digest(call_text, intent="billing", escalation=0)
rid = save_result(res, call_text)
print("انحفظ برقم:", rid)

display(pending())

# اعتماد كما هو:
# approve(rid)
# اعتماد بعد التعديل:
# approve(rid, edited_resolution="1. ...\n2. ...", notes="عدّلنا الخطوة الثانية")
# رفض:
# reject(rid, notes="الحل غير دقيق")